# VietnamNet article classification with PhoBERT

PhoBERT classifies 19 news categories using all 207,375 eligible articles from the 212,705-record corpus. The source is [VietnamNet News on student Huỳnh Thành Đạt's Hugging Face repository](https://huggingface.co/datasets/dathuynh1108/vietnamnet-news).


## 1. Model and experimental settings

The experiment uses PhoBERT Base v2, data seed 2026 and model seed 42.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import importlib.metadata


In [ ]:
requested_root = os.getenv("CO5177_REPO_ROOT")
if requested_root:
    os.chdir(Path(requested_root).expanduser().resolve())
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
if not Path("configs/text.json").exists():
    checkout = Path("/content/CO5177-PFDAV")
    if not checkout.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/dathuynh1108/CO5177-PFDAV.git",
                str(checkout),
            ],
            check=True,
        )
    os.chdir(checkout)
REPO = Path.cwd().resolve()


In [ ]:
required = [
    "numpy",
    "pandas",
    "sklearn",
    "pyarrow",
    "matplotlib",
    "torch",
    "transformers",
    "pyvi",
    "accelerate",
    "seaborn",
]
if any(importlib.util.find_spec(name) is None for name in required):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            "requirements-studies.txt",
            "accelerate>=1.2,<2",
            "seaborn",
        ],
        check=True,
    )
sys.path.insert(0, str(REPO / "src"))
print("Study checkout:", REPO)


In [ ]:
import hashlib
import importlib.metadata
import json
import math
import random
import re
import time
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from io import BytesIO

from IPython.display import Image, display
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar
from scipy.special import softmax, logsumexp
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from studies.text import prepare, prepare_inputs, split_indices
from studies.evidence import sha256


In [ ]:
config = json.loads((REPO / "configs/text.json").read_text())
SEED, MODEL_SEED = config["split_seed"], config["model_seed"]
MODEL_NAME, MODEL_REVISION = config["phobert"], config["model_revision"]
MAX_LENGTH, EPOCHS = 256, 7
BODY_HEAD_WORDS, BODY_TAIL_WORDS = 220, 80
LEARNING_RATE, WEIGHT_DECAY = 2e-5, 0.01


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path(
    os.getenv(
        "CO5177_TEXT_OUTPUT_ROOT",
        str(REPO / "artifacts/notebook-runs/text-full" / stamp),
    )
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(MODEL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"


In [ ]:
def write_json(name, value):
    (RUN_DIR / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=130, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))


In [ ]:
print(
    f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | {DEVICE}: {GPU_NAME}"
)
print(
    f"Source {config['dataset']}@{config['revision']} | model {MODEL_NAME}@{MODEL_REVISION}"
)
print(f"All eligible articles | maximum length {MAX_LENGTH} | maximum epochs {EPOCHS}")
print("Run directory:", RUN_DIR)
if DEVICE == "cpu":
    print("WARNING: full-corpus PhoBERT fine-tuning on CPU can take a long time.")


## 2. Source data and exploratory analysis

Empty headlines and duplicate records are removed; duplicate groups with conflicting categories are excluded.


In [ ]:
cohort, data_audit = prepare(config)
if "content" not in cohort:
    raise ValueError("Full-corpus preparation must retain article content")
if cohort["id"].duplicated().any():
    raise ValueError("Duplicate article IDs after preparation")
cohort = cohort.sort_values("id").reset_index(drop=True)
cohort, input_audit = prepare_inputs(cohort, config)
write_json("input_audit.json", input_audit)
classes = sorted(cohort["category"].unique())
label2id = {label: i for i, label in enumerate(classes)}
cohort["label_id"] = cohort["category"].map(label2id)
write_json("data_audit.json", data_audit)
print(json.dumps(data_audit, ensure_ascii=False, indent=2))


### Category balance and article lengths


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
pd.Series(data_audit["raw_class_counts"]).sort_index().plot.barh(
    ax=axes[0], color="#52796f"
)
axes[0].set(title="Source category counts", xlabel="Articles", ylabel="Category")
cohort.category.value_counts().sort_index().plot.barh(ax=axes[1], color="#84a98c")
axes[1].set(
    title="All eligible articles after cleaning", xlabel="Articles", ylabel="Category"
)
body_lengths = cohort.content.fillna("").map(lambda text: len(str(text).split()))
axes[2].hist(body_lengths, bins=60, color="#52796f")
axes[2].set(title="Body lengths before truncation", xlabel="Words", ylabel="Articles")
plt.tight_layout()
fig.savefig(RUN_DIR / "eda.png", dpi=160)
show_figure(fig)
plt.close(fig)


## 3. Shared split and article inputs

Grouped stratification approximates 70/15/15 while keeping equal model inputs together across both studies.


In [ ]:
partitions = split_indices(cohort, config)
train_idx = partitions["train"]
val_idx = partitions["validation"]
test_idx = partitions["test"]
cohort["split"] = ""
for name, positions in partitions.items():
    cohort.loc[positions, "split"] = name
split_ids = {
    name: sorted(cohort.iloc[positions]["id"].tolist())
    for name, positions in partitions.items()
}
split_hashes = {
    name: hashlib.sha256("\n".join(ids).encode()).hexdigest()
    for name, ids in split_ids.items()
}
split_audit = {
    "method": "20-fold StratifiedGroupKFold; 14/3/3 folds",
    "seed": SEED,
    "grouping": "Final headline or Head-Tail article token identity",
    "rows": {name: len(positions) for name, positions in partitions.items()},
    "groups": {
        name: int(cohort.iloc[positions]["input_group"].nunique())
        for name, positions in partitions.items()
    },
    "class_counts": {
        name: cohort.iloc[positions]["category"].value_counts().sort_index().to_dict()
        for name, positions in partitions.items()
    },
}
write_json("split_ids.json", split_ids)
write_json("split_hashes.json", split_hashes)
write_json("split_audit.json", split_audit)
print("Split sizes:", split_audit["rows"])
display(pd.crosstab(cohort["split"], cohort["category"]))


### Article text preparation

Inputs combine the headline with 220 initial and 80 final body words, followed by cleaning and PyVi segmentation.


In [ ]:
cohort["model_text"] = cohort["article_text"]


### Constructed model inputs


In [ ]:
if cohort.model_text.eq("").any():
    raise ValueError("Empty article input after preprocessing")
input_splits = cohort.groupby("model_text")["split"].nunique()
if input_splits.gt(1).any():
    raise ValueError(
        "Preprocessed article inputs overlap across splits; revise shared grouping before training"
    )
print("Article preprocessing completed.")


## 4. Head–Tail tokenization

Head–Tail retains 127 initial and 127 final subword tokens, plus two special tokens.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, use_fast=False
)


class NewsDataset(Dataset):
    def __init__(self, frame):
        self.labels = frame["label_id"].to_numpy(dtype=np.int64)
        self.examples, self.token_lengths = [], []
        for text in frame["model_text"]:
            ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
            self.token_lengths.append(len(ids))
            if len(ids) > MAX_LENGTH - 2:
                half = (MAX_LENGTH - 2) // 2
                ids = ids[:half] + ids[-half:]
            ids = [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
            mask = [1] * len(ids)
            pad = MAX_LENGTH - len(ids)
            ids += [tokenizer.pad_token_id] * pad
            mask += [0] * pad
            self.examples.append((torch.tensor(ids), torch.tensor(mask)))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        ids, mask = self.examples[i]
        return {
            "input_ids": ids,
            "attention_mask": mask,
            "labels": torch.tensor(self.labels[i]),
        }


### Encoded partitions


In [ ]:
train_frame = cohort[cohort.split == "train"].reset_index(drop=True)
val_frame = cohort[cohort.split == "validation"].reset_index(drop=True)
test_frame = cohort[cohort.split == "test"].reset_index(drop=True)
train_ds, val_ds, test_ds = (
    NewsDataset(train_frame),
    NewsDataset(val_frame),
    NewsDataset(test_frame),
)


### Input overlap and sequence lengths


In [ ]:
encoded_owner = {}
for name, dataset in [("train", train_ds), ("validation", val_ds), ("test", test_ds)]:
    for ids, mask in dataset.examples:
        key = hashlib.sha256(ids.numpy().tobytes()).digest()
        if key in encoded_owner and encoded_owner[key] != name:
            raise ValueError(
                "Encoded article inputs overlap across splits; revise shared grouping"
            )
        encoded_owner[key] = name
lengths = train_ds.token_lengths + val_ds.token_lengths + test_ds.token_lengths
print(
    f"Tokenized train={len(train_ds):,}, validation={len(val_ds):,}, test={len(test_ds):,}"
)
print(
    f"Subword count: median={np.median(lengths):.0f}, P95={np.quantile(lengths,.95):.0f}, truncated={(np.asarray(lengths)>MAX_LENGTH-2).mean():.1%}"
)


## 5. Full-corpus fine-tuning

PhoBERT and its 19-class sequence classification head are fine-tuned together using `AutoModelForSequenceClassification`. Training runs for up to seven epochs, with early-stopping patience three.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    num_labels=len(classes),
    id2label={i: label for i, label in enumerate(classes)},
    label2id=label2id,
)
train_counts = np.bincount(train_ds.labels, minlength=len(classes))
if (train_counts == 0).any():
    raise ValueError("A category is missing from training")
class_weights = torch.tensor(
    len(train_ds) / (len(classes) * train_counts), dtype=torch.float32
)


### Weighted loss and evaluation metrics

Cross-entropy weights use training-class frequencies: $w_c=N_{\mathrm{train}}/(19n_c)$.


In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "f1_macro": f1_score(labels, pred, average="macro"),
        "f1_weighted": f1_score(labels, pred, average="weighted"),
    }


### Optimizer and checkpoint selection


In [ ]:
gpu_memory_gib = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available()
    else 0
)
if gpu_memory_gib >= 20:
    per_device_batch = 32
elif gpu_memory_gib >= 12:
    per_device_batch = 16
elif gpu_memory_gib >= 8:
    per_device_batch = 8
else:
    per_device_batch = 4
gradient_accumulation = 64 // per_device_batch
args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=per_device_batch,
    per_device_eval_batch_size=per_device_batch * 2,
    gradient_accumulation_steps=gradient_accumulation,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    disable_tqdm=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=MODEL_SEED,
    data_seed=SEED,
    dataloader_num_workers=2 if torch.cuda.is_available() else 0,
)
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


In [ ]:
run_config = {
    "dataset": config["dataset"],
    "revision": config["revision"],
    "source_files": config["files"],
    "split_sha256": split_hashes,
    "input_preparation": input_audit,
    "split_audit": split_audit,
    "model": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "seed": SEED,
    "model_seed": MODEL_SEED,
    "max_length": MAX_LENGTH,
    "maximum_epochs": EPOCHS,
    "body_head_tail_words": [BODY_HEAD_WORDS, BODY_TAIL_WORDS],
    "class_weights_train_only": dict(zip(classes, class_weights.tolist())),
    "device": GPU_NAME,
    "training_arguments": args.to_dict(),
    "source_sha256": {
        p: sha256(REPO / p)
        for p in ["src/studies/text.py", "configs/text.json", "notebooks/text.ipynb"]
    },
    "packages": {
        p: importlib.metadata.version(p)
        for p in [
            "torch",
            "transformers",
            "accelerate",
            "numpy",
            "pandas",
            "scikit-learn",
            "pyvi",
            "pyarrow",
        ]
    },
}
write_json("run_config.json", run_config)


### Training and validation history


In [ ]:
start = time.time()
train_result = trainer.train()
train_seconds = time.time() - start
trainer.state.save_to_json(str(RUN_DIR / "trainer_state.json"))
write_json("training_log.json", trainer.state.log_history)
print(
    f"Fine-tuning time: {train_seconds/60:.1f} minutes ({train_seconds/3600:.2f} hours)"
)
print("Selected checkpoint:", trainer.state.best_model_checkpoint)
print("Completed epochs:", trainer.state.epoch)


## 6. Validation calibration and held-out evaluation

Validation selects the checkpoint, temperature and relative class factors; test evaluates the fixed decisions.


In [ ]:
val_logits = trainer.predict(val_ds).predictions
y_val = val_frame["label_id"].to_numpy()


def nll(logits, labels, temperature):
    scaled = logits / temperature
    return float(
        np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(labels)), labels])
    )


### Temperature and relative class factors


In [ ]:
cal = minimize_scalar(
    lambda t: nll(val_logits, y_val, t), bounds=(0.5, 5.0), method="bounded"
)
temperature = float(cal.x)
val_probs = softmax(val_logits / temperature, axis=1)
thresholds = np.ones(len(classes))
grid = np.linspace(0.5, 2.0, 101)
for _ in range(2):
    for c in range(len(classes)):
        best_t, best_f = thresholds[c], -1.0
        for candidate in grid:
            trial = thresholds.copy()
            trial[c] = candidate
            pred = np.argmax(val_probs / trial, axis=1)
            score = f1_score(y_val, pred, labels=[c], average=None, zero_division=0)[0]
            if score > best_f:
                best_t, best_f = float(candidate), float(score)
        thresholds[c] = best_t
print(
    f"Validation: temperature={temperature:.4f}; NLL {nll(val_logits,y_val,1):.4f} -> {nll(val_logits,y_val,temperature):.4f}"
)


### Final test predictions


In [ ]:
test_logits = trainer.predict(test_ds).predictions
y_test = test_frame["label_id"].to_numpy()
pred_before = np.argmax(test_logits, axis=1)
pred_after = np.argmax(softmax(test_logits / temperature, axis=1) / thresholds, axis=1)


def summarize(y, p):
    return {
        "accuracy": float(accuracy_score(y, p)),
        "f1_macro": float(f1_score(y, p, average="macro")),
        "f1_weighted": float(f1_score(y, p, average="weighted")),
    }


### Aggregate test results

The two decision rules use the same held-out articles. Validation determines calibration parameters.

In [ ]:
metrics = {
    "test_before_calibration": summarize(y_test, pred_before),
    "test_after_validation_calibration": summarize(y_test, pred_after),
    "calibration": {
        "temperature": temperature,
        "validation_nll_before": nll(val_logits, y_val, 1),
        "validation_nll_after": nll(val_logits, y_val, temperature),
    },
    "split_sizes": {
        "train": len(train_ds),
        "validation": len(val_ds),
        "test": len(test_ds),
    },
    "train_seconds": train_seconds,
    "epochs_completed": trainer.state.epoch,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_macro_f1": trainer.state.best_metric,
}
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("\nTest classification report after validation calibration:")
print(
    classification_report(
        y_test,
        pred_after,
        labels=np.arange(len(classes)),
        target_names=classes,
        digits=4,
        zero_division=0,
    )
)


In [ ]:
np.savez_compressed(
    RUN_DIR / "validation_logits_labels.npz",
    logits=val_logits,
    labels=y_val,
    ids=val_frame["id"].to_numpy(dtype=str),
)
np.savez_compressed(
    RUN_DIR / "test_logits_labels.npz",
    logits=test_logits,
    labels=y_test,
    ids=test_frame["id"].to_numpy(dtype=str),
)
write_json(
    "calibration.json",
    {
        "temperature": temperature,
        "class_thresholds": dict(zip(classes, thresholds.tolist())),
        "fitted_on": "validation",
    },
)


## 7. Classification results and learning curves

In [ ]:
cm = confusion_matrix(y_test, pred_after, labels=np.arange(len(classes)))
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
sns.heatmap(
    cm_norm,
    xticklabels=classes,
    yticklabels=classes,
    cmap="Blues",
    ax=axes[0],
    vmin=0,
    vmax=1,
)
axes[0].set(
    title="Normalized confusion matrix · test",
    xlabel="Predicted category",
    ylabel="True category",
)
for tick in axes[0].get_xticklabels():
    tick.set_rotation(75)
    tick.set_ha("right")
report = classification_report(
    y_test,
    pred_after,
    labels=np.arange(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)
f1s = pd.Series({c: report[c]["f1-score"] for c in classes}).sort_values()
f1s.plot.barh(ax=axes[1], color="#52796f")
axes[1].set(title="Per-category F1 · test", xlabel="F1", xlim=(0, 1))
plt.tight_layout()
confusion_fig = fig
show_figure(fig)


### Learning curves

Training loss and validation macro-F1 are plotted against optimizer steps and epochs.

In [ ]:
logs = trainer.state.log_history
train_logs = [x for x in logs if "loss" in x and "eval_loss" not in x]
eval_logs = [x for x in logs if "eval_loss" in x]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if train_logs:
    axes[0].plot(
        [x["step"] for x in train_logs],
        [x["loss"] for x in train_logs],
        color="#52796f",
    )
axes[0].set(title="Training loss by optimizer step", xlabel="Step", ylabel="Loss")
if eval_logs:
    axes[1].plot(
        [x["epoch"] for x in eval_logs],
        [x["eval_f1_macro"] for x in eval_logs],
        marker="o",
        color="#e76f51",
    )
axes[1].set(title="Validation macro-F1 by epoch", xlabel="Epoch", ylabel="Macro-F1")
plt.tight_layout()
training_fig = fig
show_figure(fig)


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
(RUN_DIR / "training_log.json").write_text(
    json.dumps(logs, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame(cm, index=classes, columns=classes).to_csv(
    RUN_DIR / "confusion_matrix_test.csv", encoding="utf-8-sig"
)
pd.DataFrame(
    classification_report(
        y_test,
        pred_after,
        labels=np.arange(len(classes)),
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )
).T.to_csv(RUN_DIR / "classification_report_test.csv", encoding="utf-8-sig")
confusion_fig.savefig(
    RUN_DIR / "test_confusion_and_class_f1.png", dpi=160, bbox_inches="tight"
)
training_fig.savefig(RUN_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
split_ids = {
    name: cohort.loc[cohort.split == name, "id"].tolist()
    for name in ["train", "validation", "test"]
}
split_hashes = {
    name: hashlib.sha256("\n".join(ids).encode()).hexdigest()
    for name, ids in split_ids.items()
}
(RUN_DIR / "split_ids.json").write_text(
    json.dumps(split_ids, indent=2), encoding="utf-8"
)
(RUN_DIR / "split_hashes.json").write_text(
    json.dumps(split_hashes, indent=2), encoding="utf-8"
)
prediction_rows = pd.DataFrame(
    {
        "id": test_frame["id"],
        "true_label": test_frame["category"],
        "predicted_before_calibration": [classes[i] for i in pred_before],
        "predicted_after_validation_calibration": [classes[i] for i in pred_after],
    }
)
prediction_rows.to_csv(
    RUN_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig"
)


In [ ]:
trainer.save_model(RUN_DIR / "best_model")
tokenizer.save_pretrained(RUN_DIR / "best_model")
print(
    f"Run configuration, metrics, logs, confusion matrix, figures and checkpoint saved to {RUN_DIR}"
)
write_json(
    "run_complete.json",
    {
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "article_count": len(cohort),
        "metrics": metrics,
    },
)


## 8. Headline baselines

Four headline classifiers use the same article IDs and partitions as the fine-tuned model. Frozen PhoBERT features use masked mean pooling; the encoder weights remain fixed.

In [ ]:
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import ConfusionMatrixDisplay
from studies.text import embeddings
from studies.evidence import metrics as classification_metrics

baseline_dir = RUN_DIR / "headline_baselines"
baseline_dir.mkdir(exist_ok=False)
texts = cohort["headline_text"].to_numpy(dtype=object)
target = cohort["label_id"].to_numpy()
validation_idx = val_idx
frame = cohort
labels = classes
OUTPUT_ROOT = baseline_dir
RUN_ID = stamp
(OUTPUT_ROOT / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + "\n"
)
print("Headline baseline directory:", baseline_dir)

In [ ]:
cache_keys_before = {
    p.name for p in (REPO / "data/text-features").glob("*") if p.is_dir()
}
phobert_features, encoding_info = embeddings(texts.tolist(), config)
cache_reused = encoding_info["feature_cache_sha256"] in cache_keys_before
run_encoding_info = {**encoding_info, "verified_embedding_cache_reused": cache_reused}
print(
    "PhoBERT:",
    {
        k: encoding_info[k]
        for k in [
            "model",
            "revision",
            "device",
            "rows",
            "max_length",
            "pooling",
            "encoding_seconds",
        ]
    },
)
print("Verified embedding cache reused:", cache_reused)
(OUTPUT_ROOT / "phobert_features.json").write_text(
    json.dumps(run_encoding_info, ensure_ascii=False, indent=2) + "\n"
)


### Validation selection

Vocabulary, IDF and classifiers fit train only. Validation macro-F1 selects $C$.

In [ ]:
records, models, fit_history = [], {}, []
for name, kind, c_values in [
    ("Majority baseline", "majority", [None]),
    ("Word TF-IDF + Linear SVM", "word", config["classifier_c"]),
    ("Character TF-IDF + Linear SVM", "character", config["classifier_c"]),
    ("Frozen PhoBERT + Logistic regression", "phobert", config["classifier_c"]),
]:
    best_score, best_model, best_c = -1.0, None, None
    trials = []
    for c in c_values:
        if kind == "majority":
            model = DummyClassifier(strategy="most_frequent")
            train_features = np.ones((len(train_idx), 1))
            validation_features = np.ones((len(validation_idx), 1))
        elif kind == "phobert":
            model = LogisticRegression(
                C=c, max_iter=2000, random_state=config["model_seed"]
            )
            train_features, validation_features = (
                phobert_features[train_idx],
                phobert_features[validation_idx],
            )
        else:
            vectorizer = TfidfVectorizer(
                analyzer="word" if kind == "word" else "char_wb",
                ngram_range=(1, 2) if kind == "word" else (3, 5),
                min_df=2,
                max_df=0.98,
                max_features=config["max_features"],
                sublinear_tf=True,
            )
            classifier = LinearSVC(
                C=c, dual="auto", max_iter=5000, random_state=config["model_seed"]
            )
            model = Pipeline([("tfidf", vectorizer), ("classifier", classifier)])
            train_features, validation_features = (
                texts[train_idx],
                texts[validation_idx],
            )
        started = time.perf_counter()
        model.fit(train_features, target[train_idx])
        elapsed = time.perf_counter() - started
        validation_prediction = model.predict(validation_features)
        score = classification_metrics(target[validation_idx], validation_prediction, labels)
        trial = {
            "C": c,
            "validation_accuracy": score["accuracy"],
            "validation_macro_f1": score["macro_f1"],
            "fit_seconds": elapsed,
        }
        trials.append(trial)
        fit_history.append({"model": name, **trial})
        print(
            f"{name} C={c}: validation macro-F1={score['macro_f1']:.4f}; fit={elapsed:.2f}s"
        )
        if score["macro_f1"] > best_score:
            best_score, best_model, best_c = score["macro_f1"], model, c
    models[name] = best_model
    records.append(
        {
            "name": name,
            "kind": kind,
            "selected_C": best_c,
            "validation_macro_f1": best_score,
            "validation_trials": trials,
        }
    )


In [ ]:
(OUTPUT_ROOT / "fit_history.json").write_text(
    json.dumps(fit_history, ensure_ascii=False, indent=2) + "\n"
)
(OUTPUT_ROOT / "splits.json").write_text(
    json.dumps(
        {
            "train": sorted(frame.iloc[train_idx]["id"].tolist()),
            "validation": sorted(frame.iloc[validation_idx]["id"].tolist()),
            "test": sorted(frame.iloc[test_idx]["id"].tolist()),
        },
        ensure_ascii=False,
    )
    + "\n"
)
split_ids = json.loads((OUTPUT_ROOT / "splits.json").read_text())
split_hashes = {
    name: hashlib.sha256("\n".join(ids).encode()).hexdigest()
    for name, ids in split_ids.items()
}
(OUTPUT_ROOT / "split_hashes.json").write_text(
    json.dumps(split_hashes, indent=2) + "\n"
)
provenance = {
    "source_sha256": {
        str(p): sha256(REPO / p)
        for p in [
            "src/studies/text.py",
            "configs/text.json",
            "notebooks/text.ipynb",
        ]
    },
    "python": sys.version,
    "packages": {
        p: importlib.metadata.version(p)
        for p in [
            "numpy",
            "pandas",
            "scikit-learn",
            "transformers",
            "torch",
            "pyvi",
            "pyarrow",
        ]
    },
}
(OUTPUT_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")


### Held-out evaluation

The selected classifiers predict the same held-out articles as the fine-tuned model.

In [ ]:
test_target = target[test_idx]
predictions = pd.DataFrame(
    {"id": frame.iloc[test_idx]["id"].to_numpy(), "target": test_target}
)
for record in records:
    name, kind = record["name"], record["kind"]
    if kind == "majority":
        test_features = np.ones((len(test_idx), 1))
    elif kind == "phobert":
        test_features = phobert_features[test_idx]
    else:
        test_features = texts[test_idx]
    predicted = models[name].predict(test_features)
    measured = classification_metrics(test_target, predicted, labels)
    record["test"] = measured
    predictions[name] = predicted
    joblib.dump(models[name], OUTPUT_ROOT / (kind + ".joblib"), compress=3)
predictions.to_csv(OUTPUT_ROOT / "predictions.csv", index=False)
summary = {
    "track": "text",
    "input_preparation": input_audit,
    "split_audit": split_audit,
    "run_id": RUN_ID,
    "seed": config["model_seed"],
    "split_seed": config["split_seed"],
    "split_sizes": {
        "train": len(train_idx),
        "validation": len(validation_idx),
        "test": len(test_idx),
    },
    "classes": labels,
    "models": records,
    "pretrained": run_encoding_info,
}
(OUTPUT_ROOT / "metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n"
)


### Comparison with the fine-tuned article model

Headline and article models differ in both input and encoder training. The comparison measures complete pipelines.

In [ ]:
comparison_rows = [
    {"Model": record["name"],
     "Accuracy": record["test"]["accuracy"],
     "Macro-F1": record["test"]["macro_f1"]}
    for record in records
]
article_metrics = json.loads((RUN_DIR / "metrics.json").read_text())
for name, key in [
    ("Fine-tuned PhoBERT article head · argmax", "test_before_calibration"),
    ("Fine-tuned PhoBERT article head · calibrated", "test_after_validation_calibration"),
]:
    score = article_metrics[key]
    comparison_rows.append({"Model": name, "Accuracy": score["accuracy"], "Macro-F1": score["f1_macro"]})
comparison = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison.round(4))
comparison.to_csv(baseline_dir / "model_comparison.csv")

In [ ]:
chosen = max(records, key=lambda record: record["validation_macro_f1"])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
model_names = [r["name"] for r in records]
x = np.arange(len(model_names))
axes[0].bar(
    x - 0.18,
    [r["test"]["accuracy"] for r in records],
    0.36,
    label="Accuracy",
    color="#263e35",
)
axes[0].bar(
    x + 0.18,
    [r["test"]["macro_f1"] for r in records],
    0.36,
    label="Macro-F1",
    color="#b74c32",
)
axes[0].set_xticks(x, model_names, rotation=25, ha="right", fontsize=8)
axes[0].set_ylim(0, 1)
axes[0].set_title("Held-out test metrics")
axes[0].legend()
ConfusionMatrixDisplay(
    np.asarray(chosen["test"]["confusion"]), display_labels=labels
).plot(ax=axes[1], colorbar=False, include_values=False, xticks_rotation=90)
axes[1].set_title("Confusion matrix: " + chosen["name"])
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "test_metrics_confusion.png", dpi=160)
show_figure(fig)
plt.close(fig)


In [ ]:
per_class = chosen["test"]["per_class"]
display(
    pd.DataFrame(
        [
            {
                "Category": label,
                "F1": per_class[label]["f1-score"],
                "Precision": per_class[label]["precision"],
                "Recall": per_class[label]["recall"],
            }
            for label in labels
        ]
    )
    .sort_values("F1")
    .head(8)
    .round(3)
)
print(
    "Predictions, classifiers, configuration, fitting history, metrics, errors and plots saved to",
    OUTPUT_ROOT,
)
(OUTPUT_ROOT / "run_complete.json").write_text(
    json.dumps(
        {
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "article_count": len(frame),
            "models": len(records),
        },
        indent=2,
    )
    + "\n"
)


## Discussion and limitations

Results cover one split and seed. The headline comparison changes both input and training method; near duplicates and temporal generalization remain unmeasured. Future experiments could compare headline-only and article inputs under matched training settings, repeat training with multiple seeds, and evaluate a temporal holdout. These extensions have not been run.

AI assistance: [AI_USE_REPORT.md](https://github.com/dathuynh1108/CO5177-PFDAV/blob/main/AI_USE_REPORT.md).
